# COM 3303 - Artificial Intelligence
## Mini Project Phase 03 - Implementation 2: Model Improvement & Optimization
### Dog vs Cat Image Classification System
**Group: ClassiFive | Rajarata University of Sri Lanka**

---

### Review of Implementation 1 Issues
| Issue | Evidence |
|-------|----------|
| Overfitting | Train acc ~98% vs Test acc ~87% (11% gap) |
| Unstable val loss | Loss curve was spiky throughout 30 epochs |
| No data augmentation | Model memorized training images |
| No early stopping | Trained all 30 epochs even after plateau |
| Fixed learning rate | Could not escape loss plateaus |

### Improvements Applied in Implementation 2
| # | Improvement | Reason |
|---|-------------|--------|
| 1 | Data Augmentation | Prevents memorization, improves generalization |
| 2 | Early Stopping | Stops training when val_loss stops improving |
| 3 | ReduceLROnPlateau | Reduces learning rate when stuck |
| 4 | Dropout after each Conv block | Reduces overfitting at feature extraction stage |
| 5 | 4th Conv block (256 filters) | Deeper network for better feature learning |
| 6 | Larger image size (128x128) | More detail preserved vs 64x64 in Impl 1 |

## SECTION 1: Import Libraries

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, classification_report

print('TensorFlow version:', tf.__version__)

## SECTION 2: Download Dataset
**Note:** Colab resets every session so we must re-download each time.
Paste your Kaggle token below before running.

In [ ]:
# ── PASTE YOUR DETAILS HERE ───────────────────────────────
KAGGLE_TOKEN    = "PASTE_YOUR_TOKEN_HERE"
KAGGLE_USERNAME = "dilshanudesh"
# ──────────────────────────────────────────────────────────

kaggle_creds = {"username": KAGGLE_USERNAME, "key": KAGGLE_TOKEN}
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump(kaggle_creds, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

!pip install -q kaggle
!kaggle datasets download -d salader/dogsvscats
!unzip -qo dogsvscats.zip -d dogs-vs-cats

print("Dataset ready!")

## SECTION 3: Dataset Setup & Verification

In [ ]:
BASE_DIR  = 'dogs-vs-cats'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
TEST_DIR  = os.path.join(BASE_DIR, 'test')

# IMPROVEMENT: Larger image size (128x128 vs 64x64 in Impl 1)
IMG_SIZE   = (128, 128)
BATCH_SIZE = 32
EPOCHS     = 30  # EarlyStopping will stop early if needed

print('Dataset Structure:')
for split, path in [('Train', TRAIN_DIR), ('Test', TEST_DIR)]:
    for cls in ['cats', 'dogs']:
        count = len(os.listdir(os.path.join(path, cls)))
        print(f'  {split} | {cls}: {count} images')

## SECTION 4: Image Preprocessing WITH Data Augmentation

**IMPROVEMENT 1: Data Augmentation**

Problem in Impl 1: No augmentation. The model saw identical images every epoch and memorized them, causing overfitting.

Fix: Apply random transformations during training so the model sees a different version of each image every epoch, forcing it to learn general features instead of memorizing pixels.

In [ ]:
# Training generator WITH augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.2,
    horizontal_flip=True,    # Mirror images left-right
    rotation_range=15,       # Rotate up to 15 degrees
    zoom_range=0.1,          # Zoom in/out by 10%
    width_shift_range=0.1,   # Shift horizontally
    height_shift_range=0.1,  # Shift vertically
    shear_range=0.1,         # Shear transformation
    fill_mode='nearest'      # Fill empty pixels after transform
)

# Test/Val: NO augmentation - only normalize
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', subset='training', shuffle=True, seed=42
)
val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', subset='validation', shuffle=False, seed=42
)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False
)

print('Class indices:', train_generator.class_indices)
print(f'Training  : {train_generator.samples} images')
print(f'Validation: {val_generator.samples} images')
print(f'Test      : {test_generator.samples} images')

## SECTION 5: Visualize Augmented Images

In [ ]:
images, labels = next(train_generator)
class_names = {v: k for k, v in train_generator.class_indices.items()}

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Augmented Training Images - Implementation 2', fontsize=14, fontweight='bold')
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    ax.set_title(class_names[int(labels[i])].capitalize())
    ax.axis('off')
plt.tight_layout()
plt.savefig('augmented_samples.png', dpi=150)
plt.show()
print('Notice: random flips, rotations and zoom applied to images')

## SECTION 6: Improved CNN Model Architecture

**IMPROVEMENT 2: Architecture Modification + Dropout Tuning**

Changes from Implementation 1:
- Added 4th convolutional block with 256 filters (deeper feature learning)
- Added Dropout(0.25) after EACH conv block (Impl 1 only had dropout at the output layer)
- BatchNormalization before Activation for more stable training
- Larger dense layer (512 neurons) with Dropout(0.5)

In [ ]:
def build_improved_model():
    model = keras.Sequential([
        layers.Input(shape=(*IMG_SIZE, 3)),

        # Block 1: 32 filters
        layers.Conv2D(32, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Block 2: 64 filters
        layers.Conv2D(64, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Block 3: 128 filters
        layers.Conv2D(128, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Block 4: 256 filters - NEW, not in Implementation 1
        layers.Conv2D(256, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Classification Head
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ], name='DogCat_CNN_v2')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

model_v2 = build_improved_model()
model_v2.summary()

## SECTION 7: Training with Improved Callbacks

**IMPROVEMENT 3: Early Stopping**
Stops training if val_loss does not improve for 5 consecutive epochs. In Impl 1, all 30 epochs ran even after the model stopped learning.

**IMPROVEMENT 4: ReduceLROnPlateau**
Automatically halves the learning rate when val_loss is stuck for 3 epochs. In Impl 1, fixed learning rate caused the spiky unstable validation loss.

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        'model_v2.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print('Starting Implementation 2 training...')
history_v2 = model_v2.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

actual_epochs = len(history_v2.history['accuracy'])
print(f'Training complete! Stopped at epoch: {actual_epochs}/{EPOCHS}')
if actual_epochs < EPOCHS:
    print('EarlyStopping triggered - prevented unnecessary overfitting!')

## SECTION 8: Plot Training Curves - Implementation 2

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_v2.history['accuracy'],     label='Train Accuracy',     color='steelblue')
ax1.plot(history_v2.history['val_accuracy'], label='Validation Accuracy', color='tomato', linestyle='--')
ax1.set_title('Training & Validation Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history_v2.history['loss'],     label='Train Loss',     color='steelblue')
ax2.plot(history_v2.history['val_loss'], label='Validation Loss', color='tomato', linestyle='--')
ax2.set_title('Training & Validation Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Implementation 2 - Improved CNN Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves_v2.png', dpi=150)
plt.show()

## SECTION 9: Evaluation on Test Set

In [ ]:
best_model_v2 = keras.models.load_model('model_v2.h5')

test_loss_v2, test_acc_v2 = best_model_v2.evaluate(test_generator, verbose=0)
print(f'Test Accuracy : {test_acc_v2 * 100:.2f}%')
print(f'Test Loss     : {test_loss_v2:.4f}')

test_generator.reset()
preds       = best_model_v2.predict(test_generator, verbose=1)
pred_labels = (preds > 0.5).astype(int).flatten()
true_labels = test_generator.classes
class_names = list(test_generator.class_indices.keys())

cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Implementation 2')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix_v2.png', dpi=150)
plt.show()

print('\nClassification Report:')
print(classification_report(true_labels, pred_labels, target_names=class_names))

## SECTION 10: Sample Predictions - Implementation 2

In [ ]:
test_generator.reset()
images, true_lbls = next(test_generator)
preds_s = best_model_v2.predict(images[:10], verbose=0)
pred_ls = (preds_s > 0.5).astype(int).flatten()
cn = {v: k for k, v in test_generator.class_indices.items()}

fig, axes = plt.subplots(2, 5, figsize=(15, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    true  = cn[int(true_lbls[i])].capitalize()
    pred  = cn[pred_ls[i]].capitalize()
    conf  = preds_s[i][0] if pred_ls[i] == 1 else 1 - preds_s[i][0]
    color = 'green' if true == pred else 'red'
    ax.set_title(f'True: {true}\nPred: {pred} ({conf*100:.1f}%)', color=color, fontsize=9)
    ax.axis('off')
plt.suptitle('Implementation 2 - Sample Predictions (Green=Correct, Red=Wrong)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_predictions_v2.png', dpi=150)
plt.show()

## SECTION 11: Quantitative Comparison - Implementation 1 vs Implementation 2

In [ ]:
# Implementation 1 known results
impl1_train = 0.98
impl1_val   = 0.85
impl1_test  = 0.871
impl1_loss  = 0.8547

# Implementation 2 results
impl2_train = history_v2.history['accuracy'][-1]
impl2_val   = history_v2.history['val_accuracy'][-1]
impl2_test  = test_acc_v2
impl2_loss  = test_loss_v2

print('=' * 62)
print(f'{"Metric":<32} {"Impl 1":>12} {"Impl 2":>12}')
print('=' * 62)
print(f'{"Train Accuracy":<32} {impl1_train*100:>11.2f}% {impl2_train*100:>11.2f}%')
print(f'{"Validation Accuracy":<32} {impl1_val*100:>11.2f}% {impl2_val*100:>11.2f}%')
print(f'{"Test Accuracy":<32} {impl1_test*100:>11.2f}% {impl2_test*100:>11.2f}%')
print(f'{"Test Loss":<32} {impl1_loss:>12.4f} {impl2_loss:>12.4f}')
print(f'{"Overfit Gap (Train-Val)":<32} {(impl1_train-impl1_val)*100:>11.2f}% {(impl2_train-impl2_val)*100:>11.2f}%')
print('=' * 62)
print(f'Test Accuracy Improvement : +{(impl2_test - impl1_test)*100:.2f}%')
print(f'Overfitting Reduced by    : {((impl1_train-impl1_val)-(impl2_train-impl2_val))*100:.2f}%')

## SECTION 12: Comparison Bar Chart

In [ ]:
metrics = ['Train Acc', 'Val Acc', 'Test Acc']
v1_vals = [impl1_train*100, impl1_val*100, impl1_test*100]
v2_vals = [impl2_train*100, impl2_val*100, impl2_test*100]

x     = range(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar([i - width/2 for i in x], v1_vals, width, label='Implementation 1 (Baseline)', color='steelblue', alpha=0.85)
b2 = ax.bar([i + width/2 for i in x], v2_vals, width, label='Implementation 2 (Improved)',  color='tomato',    alpha=0.85)

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f'{bar.get_height():.1f}%', ha='center', fontsize=10, fontweight='bold')

ax.set_ylabel('Accuracy (%)')
ax.set_title('Implementation 1 vs Implementation 2 - Accuracy Comparison', fontweight='bold')
ax.set_xticks(list(x))
ax.set_xticklabels(metrics)
ax.set_ylim(70, 108)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('comparison_chart.png', dpi=150)
plt.show()

## SECTION 13: Save Final Model

In [ ]:
best_model_v2.save('model_v2.h5')
print('Saved model versions:')
print('  model_v1.h5  -> Implementation 1 baseline model')
print('  model_v2.h5  -> Implementation 2 final improved model (FINAL)')

# Optional: Save to Google Drive so it persists after session ends
# from google.colab import drive
# drive.mount('/content/drive')
# !cp model_v2.h5 '/content/drive/MyDrive/model_v2.h5'
# print('Also saved to Google Drive!')

## SECTION 14: Predict a Single Image using Final Model

In [ ]:
from tensorflow.keras.preprocessing import image as keras_image

def predict_image(img_path, model):
    img = keras_image.load_img(img_path, target_size=IMG_SIZE)
    img_array = keras_image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array, verbose=0)[0][0]
    label      = 'Dog' if prediction > 0.5 else 'Cat'
    confidence = prediction if prediction > 0.5 else 1 - prediction

    print(f'Prediction : {label}')
    print(f'Confidence : {confidence * 100:.2f}%')
    plt.imshow(keras_image.load_img(img_path, target_size=IMG_SIZE))
    plt.title(f'Prediction: {label} ({confidence * 100:.2f}%)')
    plt.axis('off')
    plt.show()

# Usage:
# predict_image('your_image.jpg', best_model_v2)